In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)

In [4]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rating: int

In [5]:
def create_outline(state:BlogState) -> BlogState:
    # fetch title
    title=state['title']
    # call llm & generate outline
    prompt=f'Generate a detailed outline for a blog on the topic {title}'
    outline=llm.invoke(prompt).content
    # update state 
    state['outline']=outline
    
    return state

In [6]:
def create_blog(state:BlogState) -> BlogState:
    # fetch title
    title=state['title']
    # current outline
    outline=state['outline']
    # create prompt
    prompt=f'Write a detailed blog on the title - {title} using the following outline \n {outline}'
    content=llm.invoke(prompt).content
    # update state
    state['content']=content

    return state

In [15]:
def evaluate_blog(state:BlogState) -> BlogState:
    outline=state['outline']
    content=state['content']
    prompt=f'Based on the following outline \n {outline} \n Rate the following blog on scale of 1-5 \n  {content}'
    rating=llm.invoke(prompt)
    state['rating']=rating

    return state
    

In [16]:
graph=StateGraph(BlogState)

# add nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog',evaluate_blog)

# add edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate_blog')
graph.add_edge('evaluate_blog',END)

# compile graph 
workflow=graph.compile()

In [17]:
# execute the graph
initial_state={'title':'Rise of AI in India'}
final_state=workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here\'s a detailed outline for a blog post on "The Rise of AI in India," designed to be informative, engaging, and well-structured.\n\n---\n\n## Blog Post Outline: The AI Awakening: Charting the Rise of Artificial Intelligence in India\n\n**Target Audience:** Tech enthusiasts, business leaders, policymakers, students, and anyone interested in India\'s technological advancements.\n\n**Tone:** Informative, optimistic, balanced, forward-looking.\n\n**Estimated Word Count:** 1200-1800 words\n\n---\n\n### I. Catchy Title Options:\n*   The AI Awakening: Charting the Rise of Artificial Intelligence in India\n*   India\'s AI Revolution: From Silicon Valley to the Subcontinent\n*   Beyond the Hype: How AI is Reshaping India\'s Future\n*   The Next Tech Frontier: India\'s Ascent in the Global AI Landscape\n*   Decoding India\'s AI Boom: Opportunities, Challenges, and the Road Ahead\n\n### II. Introduction (Approx. 150-200 words)\n\n*   **A. Hook:** St

In [18]:
print(final_state['outline'])

Here's a detailed outline for a blog post on "The Rise of AI in India," designed to be informative, engaging, and well-structured.

---

## Blog Post Outline: The AI Awakening: Charting the Rise of Artificial Intelligence in India

**Target Audience:** Tech enthusiasts, business leaders, policymakers, students, and anyone interested in India's technological advancements.

**Tone:** Informative, optimistic, balanced, forward-looking.

**Estimated Word Count:** 1200-1800 words

---

### I. Catchy Title Options:
*   The AI Awakening: Charting the Rise of Artificial Intelligence in India
*   India's AI Revolution: From Silicon Valley to the Subcontinent
*   Beyond the Hype: How AI is Reshaping India's Future
*   The Next Tech Frontier: India's Ascent in the Global AI Landscape
*   Decoding India's AI Boom: Opportunities, Challenges, and the Road Ahead

### II. Introduction (Approx. 150-200 words)

*   **A. Hook:** Start with a compelling statement about the global AI revolution and immedia

In [19]:
print(final_state['content'])

## The AI Awakening: Charting the Rise of Artificial Intelligence in India

While the world grapples with the transformative power of Artificial Intelligence, one nation is not just observing but actively shaping its future: India. From bustling metropolises to remote villages, AI is quietly, yet profoundly, beginning to redefine how India lives, works, and innovates. With its unique blend of a vast talent pool, burgeoning digital infrastructure, massive data generation, and a myriad of complex problems ripe for technological solutions, India stands at the cusp of an AI revolution. This blog post delves into the multifaceted rise of AI in India, examining its key drivers, transformative applications across vital sectors, the pivotal role of government initiatives, the inherent challenges, and the immense future potential that positions India as a significant player in the global AI landscape.

### The Foundation: Why India is Ripe for AI

India's journey into the AI era is built upon s

In [20]:
print(final_state['rating'])

content='This blog post is an **outstanding 5 out of 5**.\n\nHere\'s a detailed breakdown of why:\n\n1.  **Adherence to Outline Structure (5/5):** The blog post follows the provided outline almost to the letter. Every major section (Introduction, Foundation, AI in Action, Government Initiatives, Challenges, Road Ahead, Conclusion) is present and in the correct order.\n2.  **Coverage of Sub-Points (5/5):** Within each section, every single sub-point listed in the outline is addressed comprehensively and thoughtfully. No key aspect is missed.\n3.  **Word Count & Pacing (5/5):** The estimated word counts for each section are remarkably well-matched. This indicates excellent planning and execution, ensuring appropriate depth for each topic without over- or under-emphasizing any part. The overall word count (approx. 1700 words) falls perfectly within the 1200-1800 word target.\n4.  **Tone and Target Audience (5/5):** The tone is consistently informative, optimistic, balanced, and forward-lo